# Week 9: Why Normalization Matters

## Learning Objectives

By the end of this notebook you will be able to:
- Explain **why** scale differences break gradient descent
- Demonstrate the gradient imbalance problem with real numbers
- Compare convergence speed with and without normalization
- Understand what normalization does to the error landscape geometrically
- Know **when NOT** to normalize
- Choose between z-score and min-max normalization

---

> **Why revisit Week 1 normalization now?**  
> In Week 1 we normalized "because it's good practice."  
> Now that you understand gradient descent, you can see the *mechanism*:  
> mismatched scales cause mismatched gradients, which cause slow or broken learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse

np.random.seed(42)
print("Libraries loaded.")

---

## Part 1: The Gradient Imbalance Problem

Gradient descent updates weights using:

$$w \leftarrow w - \alpha \cdot \frac{\partial L}{\partial w}$$

The gradient $\frac{\partial L}{\partial w_i}$ is proportional to the **scale of feature $x_i$**.  
If one feature has values in the thousands and another in the tens, their gradients differ by the same ratio.

This means:
- The large-scale feature's weight oscillates wildly (gradient too big)
- The small-scale feature's weight barely moves (gradient too small)
- You cannot pick a single learning rate that works for both

---

## Scenario 1: Age vs. Salary (~2000x scale ratio)

This is one of the most common real-world mismatches.

In [ ]:
# ── Raw data ──────────────────────────────────────────────────────────────
age    = np.array([25.0, 35.0, 45.0, 55.0, 65.0])
salary = np.array([30000.0, 50000.0, 70000.0, 90000.0, 110000.0])

print("=== Feature Statistics ===")
print(f"Age    — mean: {age.mean():.1f},  std: {age.std():.1f},  range: {age.max()-age.min():.1f}")
print(f"Salary — mean: {salary.mean():.1f}, std: {salary.std():.1f}, range: {salary.max()-salary.min():.1f}")
print(f"Scale ratio (std): {salary.std() / age.std():.0f}x")

In [ ]:
# ── Target: predict a simple score (arbitrary linear combo) ───────────────
y = 0.3 * age + 0.00002 * salary + np.random.normal(0, 0.5, size=len(age))

# ── Build design matrix X = [age, salary] ─────────────────────────────────
X_raw = np.column_stack([age, salary])

# ── Gradient descent from scratch (numpy only) ───────────────────────────
def gradient_descent(X, y, lr=1e-8, n_iter=500):
    """Simple linear regression via gradient descent. Returns weights and loss history."""
    n, p = X.shape
    w = np.zeros(p)       # start at zero
    losses = []

    for _ in range(n_iter):
        y_hat = X @ w
        residuals = y_hat - y
        loss = (residuals ** 2).mean()
        losses.append(loss)

        # gradient of MSE w.r.t. w
        grad = (2 / n) * (X.T @ residuals)
        w = w - lr * grad

    return w, losses

# Train on raw features
w_raw, losses_raw = gradient_descent(X_raw, y, lr=1e-10, n_iter=500)
print(f"Raw features  — final weights: age={w_raw[0]:.6f}, salary={w_raw[1]:.6f}")
print(f"Raw features  — final loss: {losses_raw[-1]:.6f}")

In [ ]:
# ── Z-score normalization ─────────────────────────────────────────────────
age_norm    = (age    - age.mean())    / age.std()
salary_norm = (salary - salary.mean()) / salary.std()
X_norm = np.column_stack([age_norm, salary_norm])

print("=== After Z-score Normalization ===")
print(f"Age_norm    — mean: {age_norm.mean():.4f}, std: {age_norm.std():.4f}")
print(f"Salary_norm — mean: {salary_norm.mean():.4f}, std: {salary_norm.std():.4f}")

# Train on normalized features (same lr now works for both)
w_norm, losses_norm = gradient_descent(X_norm, y, lr=0.05, n_iter=500)
print(f"\nNormalized features — final weights: age={w_norm[0]:.6f}, salary={w_norm[1]:.6f}")
print(f"Normalized features — final loss: {losses_norm[-1]:.6f}")

In [ ]:
# ── Gradient magnitude comparison ────────────────────────────────────────
n = len(age)
w_zero = np.zeros(2)

# Compute gradient at w=0 for raw features
res_raw  = X_raw  @ w_zero - y
grad_raw = (2 / n) * (X_raw.T  @ res_raw)

# Compute gradient at w=0 for normalized features
res_norm  = X_norm @ w_zero - y
grad_norm = (2 / n) * (X_norm.T @ res_norm)

print("=== Gradient Magnitudes at w=0 ===")
print(f"Raw    — grad[age]: {abs(grad_raw[0]):.2f},  grad[salary]: {abs(grad_raw[1]):.2f}")
print(f"Norm   — grad[age]: {abs(grad_norm[0]):.4f}, grad[salary]: {abs(grad_norm[1]):.4f}")
print(f"\nRaw ratio (salary grad / age grad): {abs(grad_raw[1]) / abs(grad_raw[0]):.0f}x")
print("→ Salary gradient dominates by ~2000x. One learning rate cannot serve both.")

In [ ]:
# ── Plot: loss curves and gradient bar chart ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curve: raw features
axes[0].plot(losses_raw, color='crimson', lw=2)
axes[0].set_title("Loss — Raw Features\n(very slow convergence)", fontsize=12)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("MSE Loss")
axes[0].grid(alpha=0.3)

# Loss curve: normalized features
axes[1].plot(losses_norm, color='steelblue', lw=2)
axes[1].set_title("Loss — Normalized Features\n(fast convergence)", fontsize=12)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("MSE Loss")
axes[1].grid(alpha=0.3)

# Gradient magnitude bar chart
features = ['Age\n(raw)', 'Salary\n(raw)', 'Age\n(norm)', 'Salary\n(norm)']
grads    = [abs(grad_raw[0]), abs(grad_raw[1]), abs(grad_norm[0]), abs(grad_norm[1])]
colors   = ['crimson', 'crimson', 'steelblue', 'steelblue']
bars = axes[2].bar(features, grads, color=colors, edgecolor='black', alpha=0.8)
axes[2].set_title("Gradient Magnitudes\n(salary dominates by 2000x raw)", fontsize=12)
axes[2].set_ylabel("|Gradient|")
axes[2].set_yscale('log')   # log scale to show both on same plot
axes[2].grid(alpha=0.3, axis='y')
for bar, g in zip(bars, grads):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                 f"{g:.1e}", ha='center', va='bottom', fontsize=9)

plt.suptitle("Scenario 1: Age vs Salary (2000x scale ratio)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key takeaway from Scenario 1:**  
- The salary gradient is ~2000x larger than the age gradient  
- With a learning rate large enough to move the age weight, the salary weight explodes  
- With a learning rate small enough for salary, the age weight barely moves  
- After normalization, both gradients are balanced — one learning rate works for both

---

## Scenario 2: Height (cm) vs. Weight (kg) vs. BMI

A subtler case — all three features are physiological measurements, but their scales differ by up to 6x.

In [ ]:
# ── Raw data ──────────────────────────────────────────────────────────────
height = np.array([160.0, 170.0, 175.0, 180.0, 165.0, 190.0, 155.0, 185.0])
weight = np.array([ 55.0,  70.0,  80.0,  85.0,  60.0,  95.0,  50.0,  90.0])
bmi    = np.array([ 21.5,  24.2,  26.1,  26.2,  22.0,  26.3,  20.8,  26.3])

print("=== Feature Statistics ===")
for name, feat in [("Height (cm)", height), ("Weight (kg)", weight), ("BMI", bmi)]:
    print(f"{name:15s} — mean: {feat.mean():.1f}, std: {feat.std():.2f}, range: {feat.max()-feat.min():.1f}")

print(f"\nHeight std / BMI std ratio: {height.std() / bmi.std():.1f}x")
print("→ Height gradient will be ~6x larger than BMI gradient")

In [ ]:
# ── Target: predict fitness score (synthetic) ─────────────────────────────
y2 = -0.1 * height + 0.5 * weight + 2.0 * bmi + np.random.normal(0, 0.5, size=len(height))

X2_raw  = np.column_stack([height, weight, bmi])

# Z-score normalize each feature
def zscore(arr):
    return (arr - arr.mean()) / arr.std()

X2_norm = np.column_stack([zscore(height), zscore(weight), zscore(bmi)])

# Compute gradients at w=0
n2 = len(height)
w0 = np.zeros(3)

grad2_raw  = (2/n2) * (X2_raw.T  @ (X2_raw  @ w0 - y2))
grad2_norm = (2/n2) * (X2_norm.T @ (X2_norm @ w0 - y2))

print("=== Gradient Magnitudes at w=0 ===")
labels = ['Height', 'Weight', 'BMI']
for i, lab in enumerate(labels):
    print(f"  {lab:8s} — raw: {abs(grad2_raw[i]):.4f},  normalized: {abs(grad2_norm[i]):.4f}")
print(f"\nRaw height/BMI ratio: {abs(grad2_raw[0])/abs(grad2_raw[2]):.1f}x")
print(f"Norm height/BMI ratio: {abs(grad2_norm[0])/abs(grad2_norm[2]):.2f}x  (much closer to 1)")

In [ ]:
# ── Visualize gradient magnitude bar charts ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x_pos = np.arange(3)
bar_width = 0.35

# Raw gradients
axes[0].bar(x_pos, np.abs(grad2_raw), color=['#e74c3c','#e67e22','#f1c40f'],
            edgecolor='black', alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(['Height (cm)', 'Weight (kg)', 'BMI'])
axes[0].set_title("Gradient Magnitudes — Raw Features", fontsize=12)
axes[0].set_ylabel("|Gradient|")
axes[0].grid(alpha=0.3, axis='y')
for i, g in enumerate(np.abs(grad2_raw)):
    axes[0].text(i, g + 0.5, f"{g:.2f}", ha='center', fontsize=10, fontweight='bold')

# Normalized gradients
axes[1].bar(x_pos, np.abs(grad2_norm), color=['#2ecc71','#27ae60','#1abc9c'],
            edgecolor='black', alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['Height (norm)', 'Weight (norm)', 'BMI (norm)'])
axes[1].set_title("Gradient Magnitudes — Normalized Features", fontsize=12)
axes[1].set_ylabel("|Gradient|")
axes[1].grid(alpha=0.3, axis='y')
for i, g in enumerate(np.abs(grad2_norm)):
    axes[1].text(i, g + 0.05, f"{g:.2f}", ha='center', fontsize=10, fontweight='bold')

plt.suptitle("Scenario 2: Height vs Weight vs BMI (6x imbalance)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Scenario 3: Pixel Intensity (0–255) vs. GPS Distance (0–1)

An extreme 255x ratio — the kind you see in computer vision pipelines that mix pixel values with normalized sensor readings.

In [ ]:
# ── Raw data ──────────────────────────────────────────────────────────────
pixel    = np.array([  0.0,  50.0, 128.0, 200.0, 255.0])
distance = np.array([  0.1,   0.3,   0.5,   0.8,   1.0])

print(f"Pixel    range: {pixel.max() - pixel.min():.0f}")
print(f"Distance range: {distance.max() - distance.min():.1f}")
print(f"Scale ratio: {(pixel.max()-pixel.min()) / (distance.max()-distance.min()):.0f}x")

# Synthetic target
y3 = 0.01 * pixel + 2.0 * distance

X3_raw  = np.column_stack([pixel, distance])
X3_norm = np.column_stack([zscore(pixel), zscore(distance)])

# Train with GD — raw version needs a TINY lr due to pixel scale
_, losses3_raw  = gradient_descent(X3_raw,  y3, lr=1e-7,  n_iter=300)
_, losses3_norm = gradient_descent(X3_norm, y3, lr=0.1,   n_iter=300)

print(f"\nFinal loss — raw features:        {losses3_raw[-1]:.6f}")
print(f"Final loss — normalized features: {losses3_norm[-1]:.6f}")

In [ ]:
# ── Demonstrate convergence failure at a "reasonable" lr ──────────────────
# If we use a moderate lr without normalization, it diverges
def gradient_descent_safe(X, y, lr, n_iter=300):
    """Like gradient_descent but clips loss to avoid overflow."""
    n, p = X.shape
    w = np.zeros(p)
    losses = []
    for _ in range(n_iter):
        y_hat = X @ w
        residuals = y_hat - y
        loss = (residuals ** 2).mean()
        if not np.isfinite(loss) or loss > 1e12:
            losses.append(1e12)   # diverged — cap it
            break
        losses.append(loss)
        grad = (2 / n) * (X.T @ residuals)
        w = w - lr * grad
    # pad if stopped early
    while len(losses) < n_iter:
        losses.append(losses[-1])
    return w, losses

# lr=0.001 — fine for normalized, but too large for raw pixel values
_, losses3_raw_fail  = gradient_descent_safe(X3_raw,  y3, lr=0.001, n_iter=300)
_, losses3_norm_good = gradient_descent_safe(X3_norm, y3, lr=0.1,   n_iter=300)

print("With lr=0.001 and raw features — diverged!" if losses3_raw_fail[-1] >= 1e12 else "Converged")
print(f"With lr=0.1 and normalized   — final loss: {losses3_norm_good[-1]:.6f}")

In [ ]:
# ── Plot: convergence failure vs success ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses3_raw_fail, color='crimson', lw=2, label='Raw features (lr=0.001)')
axes[0].axhline(1e12, color='black', linestyle='--', alpha=0.5, label='Divergence cap')
axes[0].set_title("Scenario 3: DIVERGENCE without normalization", fontsize=12)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("MSE Loss (capped at 1e12)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(losses3_norm_good, color='steelblue', lw=2, label='Normalized features (lr=0.1)')
axes[1].set_title("Scenario 3: Clean convergence with normalization", fontsize=12)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("MSE Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Pixel Intensity (0-255) vs GPS Distance (0-1) — 255x ratio", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 2: What Normalization Does Geometrically

Think about the **error landscape** — the surface you're trying to descend.

- **Without normalization:** features have different scales → contours are stretched ellipses  
  Gradient descent zigzags in the narrow direction and crawls in the stretched one.
  
- **With normalization:** features have equal scales → contours are circles  
  Gradient descent goes straight to the minimum.

The picture below shows this directly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Build a 2D loss surface for each case ─────────────────────────────────
w1_range = np.linspace(-3, 3, 300)
w2_range = np.linspace(-3, 3, 300)
W1, W2   = np.meshgrid(w1_range, w2_range)

# Without normalization: w2 (salary) axis is compressed → elongated ellipse
# We simulate this by scaling one axis
loss_raw  = W1**2 / 0.05 + W2**2 * 4    # elongated along W1
loss_norm = W1**2           + W2**2      # circular

levels = [0.2, 0.5, 1, 2, 3, 5, 8, 12, 18]

# Panel 1: elongated (without normalization)
cp1 = axes[0].contour(W1, W2, loss_raw, levels=levels, cmap='RdYlBu_r')
axes[0].clabel(cp1, inline=True, fontsize=8)
# Simulate zigzag path
path_raw = [(2.5, 2.5), (2.5, 0.8), (-1.5, 0.8), (-1.5, 0.3), (0.5, 0.3), (0.5, 0.1), (0.0, 0.0)]
px, py = zip(*path_raw)
axes[0].plot(px, py, 'r-o', lw=2, ms=5, label='GD path (zigzag)')
axes[0].scatter([0], [0], s=200, c='gold', zorder=5, label='Minimum')
axes[0].set_title("WITHOUT normalization\n(elongated contours → zigzag)", fontsize=12)
axes[0].set_xlabel("w₁ (age weight)")
axes[0].set_ylabel("w₂ (salary weight)")
axes[0].legend(fontsize=9)
axes[0].set_xlim(-3, 3); axes[0].set_ylim(-3, 3)
axes[0].set_aspect('equal')
axes[0].grid(alpha=0.2)

# Panel 2: circular (with normalization)
cp2 = axes[1].contour(W1, W2, loss_norm, levels=[0.1, 0.3, 0.6, 1.0, 1.5, 2.5, 4, 6], cmap='RdYlBu_r')
axes[1].clabel(cp2, inline=True, fontsize=8)
# Straight descent path
t = np.linspace(0, 1, 8)
px2 = 2.5 * (1 - t)
py2 = 2.5 * (1 - t)
axes[1].plot(px2, py2, 'b-o', lw=2, ms=5, label='GD path (straight)')
axes[1].scatter([0], [0], s=200, c='gold', zorder=5, label='Minimum')
axes[1].set_title("WITH normalization\n(circular contours → straight descent)", fontsize=12)
axes[1].set_xlabel("w₁ (age_norm weight)")
axes[1].set_ylabel("w₂ (salary_norm weight)")
axes[1].legend(fontsize=9)
axes[1].set_xlim(-3, 3); axes[1].set_ylim(-3, 3)
axes[1].set_aspect('equal')
axes[1].grid(alpha=0.2)

plt.suptitle("Error Landscape: Normalization Makes Contours Circular", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Left:  Without normalization — stretched ellipse, gradient points diagonally,")
print("       each step corrects one direction while overshooting the other (zigzag).")
print("Right: With normalization    — circular landscape, gradient points straight to")
print("       minimum, convergence in far fewer iterations.")

---

## Part 3: Types of Normalization

### Z-score (Standardization)

$$x' = \frac{x - \mu}{\sigma}$$

- Result has mean 0, std 1
- **Use when:** the distribution is roughly Gaussian, or you use gradient-based methods
- **Good for:** linear/logistic regression, neural networks, SVM, PCA

### Min-Max Normalization

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- Result is in [0, 1]
- **Use when:** you need bounded output (e.g., pixel values fed to CNN, distance metrics)
- **Sensitive to outliers** — one outlier compresses everything else

### Comparison

In [ ]:
# ── Compare z-score vs min-max on the age/salary data ─────────────────────
data = salary.copy()
# add an outlier to show sensitivity
data_with_outlier = np.append(data, 500000.0)

def minmax(arr):
    return (arr - arr.min()) / (arr.max() - arr.min())

print("=== Normalization Comparison ===")
print(f"{'Value':>10}  {'Z-score':>10}  {'Min-Max':>10}  {'Z-score+outlier':>17}  {'MinMax+outlier':>15}")
print("-" * 70)
for v, z, m, zo, mo in zip(data,
                            zscore(data), minmax(data),
                            zscore(data_with_outlier)[:5],
                            minmax(data_with_outlier)[:5]):
    print(f"{v:10.0f}  {z:10.4f}  {m:10.4f}  {zo:17.4f}  {mo:15.4f}")

print("\nObservation:")
print("Z-score: adding 500k outlier slightly shifts values but preserves spread")
print("Min-Max: adding 500k outlier COMPRESSES all original values into [0, 0.22]")
print("→ Z-score is more robust to outliers")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x_labels = [f"{int(v/1000)}k" for v in data]
x_pos = np.arange(len(data))

axes[0].bar(x_pos, data, color='salmon', edgecolor='black')
axes[0].set_title("Original Salary Values", fontsize=11)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(x_labels)
axes[0].set_ylabel("Value"); axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(x_pos, zscore(data), color='steelblue', edgecolor='black')
axes[1].set_title("Z-score Normalized\n(mean=0, std=1)", fontsize=11)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(x_labels)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_ylabel("Z-score"); axes[1].grid(alpha=0.3, axis='y')

axes[2].bar(x_pos, minmax(data), color='mediumseagreen', edgecolor='black')
axes[2].set_title("Min-Max Normalized\n(range=[0,1])", fontsize=11)
axes[2].set_xticks(x_pos); axes[2].set_xticklabels(x_labels)
axes[2].set_ylabel("Normalized Value"); axes[2].grid(alpha=0.3, axis='y')

plt.suptitle("Z-score vs Min-Max Normalization", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 4: When NOT to Normalize

Normalization is not always appropriate. Here are the key exceptions:

| Situation | Why NOT to normalize | What to do instead |
|-----------|---------------------|--------------------|
| **Categorical features** (e.g., 0=male, 1=female) | Normalization destroys the categorical meaning | One-hot encode instead |
| **Already-normalized features** (e.g., probabilities 0–1) | Double-normalizing distorts the scale | Leave as is |
| **Tree-based models** (Decision Tree, Random Forest, XGBoost) | Trees split on thresholds — scale doesn't affect splits | Skip normalization |
| **Interpretability matters** | Z-scores are harder to explain to non-technical stakeholders | Keep original scale, use robust standardization |
| **Sparse features** (mostly zeros) | Z-score normalization destroys sparsity | Use MaxAbsScaler or leave sparse |

### Quick decision flowchart:

```
Is feature categorical? ──YES──→ One-hot encode, don't normalize
         │
         NO
         │
Are you using a tree model? ──YES──→ Normalization optional (won't hurt, doesn't help)
         │
         NO
         │
Do features have very different scales? ──YES──→ NORMALIZE
         │
         NO
         │
         → Normalization optional
```

In [ ]:
# ── Demonstrate: tree model is scale-invariant ────────────────────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Simple binary classification with mismatched scales
np.random.seed(0)
n_samples = 100
age_cls    = np.random.uniform(20, 70, n_samples)
income_cls = np.random.uniform(20000, 120000, n_samples)
# Rule: high earner OR old → class 1
y_cls = ((income_cls > 70000) | (age_cls > 50)).astype(int)

X_cls_raw  = np.column_stack([age_cls, income_cls])
X_cls_norm = np.column_stack([zscore(age_cls), zscore(income_cls)])

dt_raw  = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_cls_raw,  y_cls)
dt_norm = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_cls_norm, y_cls)

print("Decision Tree accuracy on raw features:        ", accuracy_score(y_cls, dt_raw.predict(X_cls_raw)))
print("Decision Tree accuracy on normalized features: ", accuracy_score(y_cls, dt_norm.predict(X_cls_norm)))
print("\n→ Same accuracy — trees are scale invariant! Normalization doesn't help them.")

---

## Summary

| Concept | Key Point |
|---------|----------|
| **Why normalization matters** | Gradient magnitude ∝ feature scale — mismatched scales → imbalanced gradients |
| **Scenario 1 (Age vs Salary)** | ~2000x ratio → salary gradient dominates, age weight barely moves |
| **Scenario 2 (Height/Weight/BMI)** | ~6x ratio → height gradient 6x larger than BMI |
| **Scenario 3 (Pixel vs GPS)** | 255x ratio → divergence at any practical learning rate |
| **Geometric insight** | Normalization turns elongated ellipse contours into circles → straight-line descent |
| **Z-score** | Best default for gradient-based models; robust to distribution shape |
| **Min-Max** | Best when you need [0,1] output; sensitive to outliers |
| **Don't normalize** | Categorical features, tree models, already-normalized features |

---

**Next Week:** We'll see what happens when linear models are simply not powerful enough — and how polynomial features and decision trees escape that limitation.